In [ ]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

## Samm 1: Määra Pydantic mudelid struktureeritud väljundite jaoks

Need mudelid määratlevad **skeemi**, mida agendid tagastavad. `response_format` kasutamine Pydanticuga tagab:
- ✅ Tüübiohutuse andmete eraldamisel
- ✅ Automaatse valideerimise
- ✅ Vaba teksti vastustest tulenevate analüüsivigade puudumise
- ✅ Lihtsa tingimusliku marsruutimise väljade põhjal


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## Samm 2: Loo hotelli broneerimise tööriist

See tööriist on see, mida **availability_agent** kutsub, et kontrollida, kas toad on saadaval. Kasutame `@ai_function` dekrataatorit, et:
- Muuta Python funktsioon AI-kõneldavaks tööriistaks
- Automaatselt genereerida JSON skeem LLM-ile
- Käsitleda parameetrite valideerimist
- Võimaldada agendi automaatset käivitust

Selle demo jaoks:
- **Stockholm, Seattle, Tokyo, London, Amsterdam** → On toad saadaval ✅
- **Kõik teised linnad** → Toasid pole saadaval ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## Samm 3: Defineerige tingimuse funktsioonid marsruutimiseks

Need funktsioonid kontrollivad agendi vastust ja määravad, millist teed töövoos valida.

**Oluline muster:**
1. Kontrollige, kas sõnum on `AgentExecutorResponse`
2. Parsige struktureeritud väljund (Pydantic mudel)
3. Tagastage `True` või `False`, et suunamist kontrollida

Töövoog hindab neid tingimusi **servadel**, et otsustada järgmiseks kutsutav täitja.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## Samm 4: Loo kohandatud kuvamisega täitja

Täitjad on töövoo komponendid, mis teostavad transformatsioone või kõrvalmõjusid. Kasutame `@executor` dekoratiivset funktsiooni, et luua kohandatud täitja, mis kuvab lõpptulemuse.

**Põhikontseptsioonid:**
- `@executor(id="...")` - Registreerib funktsiooni töövoo täitjana
- `WorkflowContext[Never, str]` - Tüübi vihjed sisendi/väljundi jaoks
- `ctx.yield_output(...)` - Annab töövoo lõpptulemuse


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

## Samm 5: Keskkonnamuutujate laadimine

Seadista LLM klient. See näide töötab järgmistega:
- **GitHub mudelid** (Tasuta tase GitHubi tokeniga)
- **Azure OpenAI**
- **OpenAI**


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 6. samm: Loo tehisintellekti agendid struktureeritud väljunditega

Loome **kolm spetsialiseerunud agenti**, igaüks ümbritsetud `AgentExecutor` abil:

1. **availability_agent** - Kontrollib hotelli saadavust tööriista abil
2. **alternative_agent** - Pakub alternatiivseid linnu (kui tube ei ole)
3. **booking_agent** - Julgustab broneerima (kui tube on saadaval)

**Põhijooned:**
- `tools=[hotel_booking]` - Annab agentidele tööriista
- `response_format=PydanticModel` - Sundib struktureeritud JSON-väljundit
- `AgentExecutor(..., id="...")` - Mähib agendi töövoo kasutamiseks


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    await provider.create_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    await provider.create_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    await provider.create_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## Samm 7: Koosta töövoog tingimuslike servadega

Nüüd kasutame `WorkflowBuilder`, et ehitada graaf tingimusliku marsruutimisega:

**Töövoo struktuur:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**Põhimetoodikad:**
- `.set_start_executor(...)` - Määrab lähtepunkti
- `.add_edge(from, to, condition=...)` - Lisab tingimusliku serva
- `.build()` - Viimistleb töövoo


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## Samm 8: Käivita testjuhtum 1 - Linn PUUDUB saadavus (Pariis)

Testime **puuduva saadavuse** teed, pärides hotelle Pariisis (millel meie simulatsioonis pole toad saadaval).


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## Samm 9: Käivita testjuhtum 2 - Linn KOGUVAÕLUGA (Stockholm)

Nüüd testime **saadavuse** rada, pärides hotelle Stockholmis (kus on meie simulatsioonis toad saadaval).


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## Peamised järeldused ja järgmised sammud

### ✅ Mida sa õppisid:

1. **WorkflowBuilder mustrit**
   - Kasuta `.set_start_executor()` sisendpunkti määramiseks
   - Kasuta `.add_edge(from, to, condition=...)` tingimuslikuks marsruutimiseks
   - Kutsu `.build()` töövoo lõpetamiseks

2. **Tingimuslik marsruutimine**
   - Tingimusfunktsioonid kontrollivad `AgentExecutorResponse` objekti
   - Töötle struktureeritud väljundeid marsruutimisotsuste tegemiseks
   - Tagasta `True`, et aktiveerida serv, `False` selle vahelejätmiseks

3. **Tööriistade integreerimine**
   - Kasuta `@ai_function`, et muuta Python funktsioonid AI tööriistadeks
   - Agendid kutsuvad tööriistu automaatselt vajadusel
   - Tööriistad tagastavad JSON-i, mida agendid suudavad parsida

4. **Struktureeritud väljundid**
   - Kasuta Pydantic mudelid tüübiturvaliseks andmete väljavõtmiseks
   - Määra `response_format=MyModel` agentide loomisel
   - Parsige vastuseid `Model.model_validate_json()` abil

5. **Kohandatud täitjad**
   - Kasuta `@executor(id="...")`, et luua töövoo komponente
   - Täitjad võivad andmeid muuta või teha kõrvalmõjusid
   - Kasuta `ctx.yield_output()`, et esitada töövoo tulemusi

### 🚀 Reaalse maailma rakendused:

- **Reisibroneering**: kontrolli saadavust, paku alternatiive, võrdle valikuid
- **Klienditeenindus**: marsruudi määramine probleemi tüübi, meeleolu, prioriteedi alusel
- **E-kaubandus**: inventuuri kontroll, alternatiivide pakkumine, tellimuste töötlemine
- **Sisumodereerimine**: marsruudi määramine toksiilsuspunktide ja kasutaja märguannete alusel
- **Kinnituse töövood**: marsruudi määramine summa, kasutaja rolli, riskitaseme järgi
- **Mitmeastmeline töötlemine**: marsruudi määramine andmete kvaliteedi ja täielikkuse põhjal

### 📚 Järgmised sammud:

- Lisa keerukamaid tingimusi (mitme kriteeriumiga)
- Rakenda tsüklid töövoo seisundi haldamisega
- Lisa alam-töövood taaskasutatavate komponentide jaoks
- Integreeri reaalsete API-dega (hotellibroneeringud, inventuuri süsteemid)
- Lisa veakäsitus ja tagavarateed
- Visualiseeri töövooge sisseehitatud visualiseerimistööriistadega


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Lahtiütlus**:
See dokument on tõlgitud kasutades AI tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi me püüdleme täpsuse poole, palun pange tähele, et automatiseeritud tõlgetes võib esineda vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlkega seotud eksimustest või valesti mõistmistest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
